In [1]:
import os
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

In [2]:
# Define paths for casting data
train_dir = '../data/casting_data/train'
test_dir = '../data/casting_data/test'

In [3]:
# Check if directories exist
print(f"Train directory exists: {os.path.exists(train_dir)}")
print(f"Test directory exists: {os.path.exists(test_dir)}")

Train directory exists: True
Test directory exists: True


In [4]:
# Define image transformations (Resize to 224x224 and convert to tensors)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [5]:
# Load datasets using ImageFolder
train_dataset = ImageFolder(root=train_dir, transform=transform)
test_dataset = ImageFolder(root=test_dir, transform=transform)

In [6]:
# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [7]:
print("Casting datasets and DataLoaders created successfully!")
print(f"Classes found: {train_dataset.classes}")
print(f"Total training images: {len(train_dataset)}")
print(f"Total test images: {len(test_dataset)}")

Casting datasets and DataLoaders created successfully!
Classes found: ['defective', 'normal']
Total training images: 6633
Total test images: 715


In [8]:
import torch.nn as nn
import torch.nn.functional as F

# Define the CNN Model for Casting Defect Detection
class CastingCNN(nn.Module):
    def __init__(self):
        super(CastingCNN, self).__init__()
        
        # Convolutional Layers
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        
        # Pooling layer
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # Fully Connected Layers
        # Input image size: 224x224 -> after 3 max pools (224 -> 112 -> 56 -> 28)
        self.fc1 = nn.Linear(64 * 28 * 28, 128)
        self.fc2 = nn.Linear(128, 2)  # 2 classes: defective and normal
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        
        # Flatten the tensor for fully connected layers
        x = x.view(-1, 64 * 28 * 28)
        
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        
        return x

In [9]:
# Initialize the model
cnn_model = CastingCNN()
print("CNN Model created successfully!")
print(cnn_model)

CNN Model created successfully!
CastingCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=50176, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=2, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)


In [10]:
import torch.optim as optim

#Define Loss Function and Optimizer for Classification
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

#Training Loop (Running for 3 epochs as a start)
num_epochs = 3
cnn_model.train()

CastingCNN(
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=50176, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=2, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
)

In [11]:
print("CNN Training started...")
for epoch in range(num_epochs):
    epoch_loss = 0.0
    correct = 0
    total = 0
    
    for batch_images, batch_labels in train_loader:
        # Zero the gradients
        optimizer.zero_grad()
        
        # Forward pass
        outputs = cnn_model(batch_images)
        
        # Calculate loss
        loss = criterion(outputs, batch_labels)
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        # Calculate accuracy
        _, predicted = torch.max(outputs.data, 1)
        total += batch_labels.size(0)
        correct += (predicted == batch_labels).sum().item()
        
    epoch_accuracy = 100 * correct / total
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss/len(train_loader):.4f}, Accuracy: {epoch_accuracy:.2f}%")

print("CNN Model training completed successfully!")

CNN Training started...
Epoch [1/3], Loss: 0.3403, Accuracy: 84.17%
Epoch [2/3], Loss: 0.1003, Accuracy: 96.35%
Epoch [3/3], Loss: 0.0631, Accuracy: 97.68%
CNN Model training completed successfully!


In [12]:
#Evaluate the CNN model on test dataset
cnn_model.eval()
test_correct = 0
test_total = 0

with torch.no_grad():
    for batch_images, batch_labels in test_loader:
        outputs = cnn_model(batch_images)
        _, predicted = torch.max(outputs.data, 1)
        test_total += batch_labels.size(0)
        test_correct += (predicted == batch_labels).sum().item()

test_accuracy = 100 * test_correct / test_total
print(f"Test Evaluation Completed! Test Accuracy: {test_accuracy:.2f}%")

Test Evaluation Completed! Test Accuracy: 97.34%


In [13]:
#Save the trained CNN model weights
import os
os.makedirs('../models', exist_ok=True)
model_save_path = '../models/casting_cnn_model.pth'
torch.save(cnn_model.state_dict(), model_save_path)

print(f"CNN Model successfully saved to {model_save_path}")

CNN Model successfully saved to ../models/casting_cnn_model.pth
